In [11]:
import pandas as pd
import numpy as np

import joblib
import json

from sklearn.preprocessing import LabelEncoder
from sklearn.metrics.pairwise import cosine_similarity

In [12]:
df = pd.read_csv(
    "../datasets/cleaned/startup_info.csv",
    low_memory=False
)

print(df.shape)

(50000, 151)


In [13]:
similarity_features = [

    "sector",
    "sub_sector",
    "technology_domain",
    "startup_stage",
    "market_type"
]

In [14]:
encoders = {}

for col in similarity_features:

    le = LabelEncoder()

    df[col] = le.fit_transform(
        df[col].astype(str)
    )

    encoders[col] = le

In [15]:
X = df[[
    "sector",
    "sub_sector",
    "technology_domain",
    "startup_stage",
    "market_type",

    "funding_strength_score",

    "market_opportunity_score",

    "growth_score",

    "innovation_score"
]]

In [16]:
from sklearn.neighbors import NearestNeighbors

In [17]:
nn_model = NearestNeighbors(

    n_neighbors=6,

    metric="cosine"

)

nn_model.fit(X)

NearestNeighbors(metric='cosine', n_neighbors=6)

In [20]:
def find_competitors(
    startup_index,
    top_n=5
):

    distances, indices = nn_model.kneighbors(

        X.iloc[[startup_index]],

        n_neighbors=top_n+1
    )

    results = []

    for idx, dist in zip(

        indices[0][1:],

        distances[0][1:]

    ):

        similarity = round(

            (1-dist)*100,

            2
        )

        results.append({

            "startup_name":
                original_df.iloc[idx]["startup_name"],

            "similarity":
                similarity
        })

    return results

In [21]:
original_df = pd.read_csv(
    "../datasets/cleaned/startup_info.csv",
    low_memory=False
)

In [22]:
sample = find_competitors(0)

print(sample)

[{'startup_name': 'DriveMax Analytics', 'similarity': 100.0}, {'startup_name': 'GrowthVertex Systems', 'similarity': 100.0}, {'startup_name': 'OrbitPro Technologies', 'similarity': 100.0}, {'startup_name': 'SigmaMax Technologies', 'similarity': 99.99}, {'startup_name': 'MoneyRise Innovations', 'similarity': 99.99}]


In [24]:
engine = {

    "engine_name":
        "Competitor Discovery Engine",

    "version":
        "V1"
}

joblib.dump(

    engine,

    "../models/competitor_discovery_engine/competitor_discovery_engine.pkl"
)

['../models/competitor_discovery_engine/competitor_discovery_engine.pkl']

In [25]:
joblib.dump(

    encoders,

    "../models/competitor_discovery_engine/encoders.pkl"
)

['../models/competitor_discovery_engine/encoders.pkl']

In [26]:
metadata = {

    "engine":
        "Competitor Discovery",

    "similarity_method":
        "Cosine Similarity",

    "features":
        similarity_features
}

with open(

    "../models/competitor_discovery_engine/metadata.json",

    "w"

) as f:

    json.dump(
        metadata,
        f,
        indent=4
    )

In [28]:
print(find_competitors(0))

print(find_competitors(100))

print(find_competitors(500))

[{'startup_name': 'DriveMax Analytics', 'similarity': 100.0}, {'startup_name': 'GrowthVertex Systems', 'similarity': 100.0}, {'startup_name': 'OrbitPro Technologies', 'similarity': 100.0}, {'startup_name': 'SigmaMax Technologies', 'similarity': 99.99}, {'startup_name': 'MoneyRise Innovations', 'similarity': 99.99}]
[{'startup_name': 'TechWave Networks', 'similarity': 100.0}, {'startup_name': 'MatrixFlowX Solutions', 'similarity': 100.0}, {'startup_name': 'SecureLogic Technologies', 'similarity': 100.0}, {'startup_name': 'SecureNext Solutions', 'similarity': 100.0}, {'startup_name': 'SkyLeap Ventures', 'similarity': 100.0}]
[{'startup_name': 'StackIndia Innovations', 'similarity': 100.0}, {'startup_name': 'PixelAI Solutions', 'similarity': 99.99}, {'startup_name': 'AutoMind Networks', 'similarity': 99.99}, {'startup_name': 'RiseCapital Systems', 'similarity': 99.99}, {'startup_name': 'WorkSyncX Technologies', 'similarity': 99.99}]


In [29]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.neighbors import NearestNeighbors

In [30]:
cat_cols = [
    "sector",
    "sub_sector",
    "technology_domain",
    "startup_stage"
]

In [31]:
num_cols = [
    "funding_strength_score",
    "growth_score",
    "market_opportunity_score",
    "innovation_score",
    "startup_health_score"
]

In [32]:
preprocessor = ColumnTransformer(
    transformers=[
        ("cat", OneHotEncoder(handle_unknown="ignore"), cat_cols),
        ("num", "passthrough", num_cols)
    ]
)

X_transformed = preprocessor.fit_transform(original_df)

In [33]:
nn_model = NearestNeighbors(
    n_neighbors=6,
    metric="cosine"
)

nn_model.fit(X_transformed)

NearestNeighbors(metric='cosine', n_neighbors=6)

In [34]:
def find_competitors(startup_index, top_n=5):

    row = X_transformed[startup_index]

    distances, indices = nn_model.kneighbors(
        row,
        n_neighbors=top_n+1
    )

    results = []

    for idx, dist in zip(
        indices[0][1:],
        distances[0][1:]
    ):

        results.append({
            "startup_name":
                original_df.iloc[idx]["startup_name"],

            "similarity":
                round((1-dist)*100,2)
        })

    return results

In [36]:
engine = {

    "engine_name":
        "Competitor Discovery Engine",

    "version":
        "V1"
}

joblib.dump(

    engine,

    "../models/competitor_discovery_engine/competitor_discovery_engine.pkl"
)

['../models/competitor_discovery_engine/competitor_discovery_engine.pkl']

In [37]:
print("Startup:")
print(original_df.iloc[0][[
    "startup_name",
    "sector",
    "sub_sector",
    "technology_domain"
]])

print("\nCompetitors:")
for c in find_competitors(0):
    print(c)

Startup:
startup_name         HyperAnalytics Technologies
sector                                    EdTech
sub_sector                      Fleet Management
technology_domain                          AR/VR
Name: 0, dtype: object

Competitors:
{'startup_name': 'AudioGo Analytics', 'similarity': 99.87}
{'startup_name': 'RideIndia Technologies', 'similarity': 99.86}
{'startup_name': 'DriveMax Analytics', 'similarity': 99.85}
{'startup_name': 'QuantumMatrix Ventures', 'similarity': 99.84}
{'startup_name': 'AgriLoop Private Limited', 'similarity': 99.84}
